# Day 2 – Module 1: Open-Source AI Ecosystem

**Topics covered:**
- Role of open-source AI in enterprise environments
- Overview of the Hugging Face ecosystem (Hub, transformers, datasets)
- Models, datasets, and community resources

**What you will do in this notebook:**
Understand why enterprises choose open-source AI models, explore the Hugging Face ecosystem hands-on — browse models, load a pre-trained model for inference, load and inspect a dataset — and build a reference for the libraries you will use throughout Day 2. Every code cell runs against real Hugging Face resources; model names and settings are loaded from `config.json`.

## 1. The Hugging Face ecosystem — architecture overview

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    HUGGING FACE ECOSYSTEM                               │
│                                                                         │
│  ┌──────────────────────────────────────────────────────────────────┐   │
│  │                     HUGGING FACE HUB                              │   │
│  │                                                                    │   │
│  │   ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐   │   │
│  │   │  Models   │    │ Datasets │    │  Spaces  │    │ Model    │   │   │
│  │   │  500K+    │    │  100K+   │    │  (Demos) │    │ Cards    │   │   │
│  │   │          │    │          │    │          │    │ +License │   │   │
│  │   └────┬─────┘    └────┬─────┘    └──────────┘    └──────────┘   │   │
│  │        │               │                                          │   │
│  └────────┼───────────────┼──────────────────────────────────────────┘   │
│           │               │                                              │
│   ┌───────▼───────┐ ┌────▼────────┐                                    │
│   │ transformers  │ │  datasets   │   YOUR CODE                        │
│   │               │ │             │                                      │
│   │ • pipeline()  │ │ • load_     │   • Load model from Hub             │
│   │ • AutoModel   │ │   dataset() │   • Run inference locally           │
│   │ • AutoToken.  │ │ • map()     │   • Load and process data           │
│   │ • Trainer     │ │ • filter()  │   • Fine-tune (optional)            │
│   └───────────────┘ └─────────────┘                                     │
│                                                                         │
│   Also: sentence-transformers, tokenizers, PEFT, accelerate             │
└─────────────────────────────────────────────────────────────────────────┘
```

| Component | What it provides | Day 2 usage |
|-----------|-----------------|-------------|
| Hub | Model/dataset hosting, versioning, model cards | Browse, download models and datasets |
| transformers | Load/run models, tokenizers, pipelines | Summarization, QA, sentiment, classification |
| datasets | Load/process/stream datasets | Load benchmark datasets for evaluation |
| sentence-transformers | Embed text into vectors | Embeddings for semantic search (Module 03) |

## 2. Setup

Load environment and configuration. Model names come from `config.json`. Public models are downloaded from the Hub without login — no extra secrets required for this module.

In [1]:
import json
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

with open(Path.cwd() / "config.json") as f:
    CONFIG = json.load(f)

print("config.json loaded:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

config.json loaded:
  embedding_model: all-MiniLM-L6-v2
  summarization_model: Falconsai/text_summarization
  qa_model: distilbert-base-cased-distilled-squad
  sentiment_model: distilbert-base-uncased-finetuned-sst-2-english
  image_classification_model: google/vit-base-patch16-224
  faiss_index_type: IndexFlatL2
  chroma_collection_name: course_demo
  top_k: 3


In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

import torch
import transformers
import datasets as ds_lib

print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"datasets:     {ds_lib.__version__}")
print(f"Device:       {'cuda' if torch.cuda.is_available() else 'cpu'}")

torch:        2.10.0+cpu
transformers: 4.57.6
datasets:     2.21.0
Device:       cpu


## 3. Role of open-source AI in enterprise environments

**Why enterprises adopt open-source AI models:**

| Factor | API-based (e.g. OpenAI) | Open-source (e.g. Hugging Face) |
|--------|------------------------|---------------------------------|
| **Cost** | Per-token pricing; scales linearly | Compute cost only; amortized at scale |
| **Data sovereignty** | Data sent to third-party servers | Runs on-prem or in private cloud |
| **Customization** | Prompt engineering, fine-tune via API | Full model access: fine-tune, prune, distill |
| **Audit / compliance** | Black-box model weights | Inspect weights, gradients, tokenizer |
| **Vendor lock-in** | Tied to provider's API contract | Swap models freely (same pipeline API) |
| **Ops burden** | Managed by provider | You manage infra (GPU, scaling, updates) |

**Enterprise scenarios where open-source is preferred:**

In [3]:
# Scenario examples — not code to run, but context for the patterns below
scenarios = [
    {
        "scenario": "Data sovereignty (healthcare, finance)",
        "requirement": "Patient records / financial data must not leave private infrastructure",
        "open_source_approach": "Run model on-prem with transformers; no external API calls",
    },
    {
        "scenario": "Domain fine-tuning (legal, medical)",
        "requirement": "Generic model struggles with domain jargon and classification",
        "open_source_approach": "Fine-tune open model on proprietary labeled data",
    },
    {
        "scenario": "High-volume inference (10M+ requests/day)",
        "requirement": "Per-token API cost exceeds $50K/month at scale",
        "open_source_approach": "Self-host on GPU cluster; fixed compute cost",
    },
    {
        "scenario": "Regulatory audit (EU AI Act, SOC 2)",
        "requirement": "Must explain model decisions and demonstrate fairness testing",
        "open_source_approach": "Full access to model weights, tokenizer, training data provenance",
    },
]

for s in scenarios:
    print(f"Scenario:  {s['scenario']}")
    print(f"  Need:    {s['requirement']}")
    print(f"  Approach: {s['open_source_approach']}")
    print()

Scenario:  Data sovereignty (healthcare, finance)
  Need:    Patient records / financial data must not leave private infrastructure
  Approach: Run model on-prem with transformers; no external API calls

Scenario:  Domain fine-tuning (legal, medical)
  Need:    Generic model struggles with domain jargon and classification
  Approach: Fine-tune open model on proprietary labeled data

Scenario:  High-volume inference (10M+ requests/day)
  Need:    Per-token API cost exceeds $50K/month at scale
  Approach: Self-host on GPU cluster; fixed compute cost

Scenario:  Regulatory audit (EU AI Act, SOC 2)
  Need:    Must explain model decisions and demonstrate fairness testing
  Approach: Full access to model weights, tokenizer, training data provenance



## 4. Models on the Hugging Face Hub

The Hub hosts 500K+ models organized by **task** (summarization, classification, translation, ...), **framework** (PyTorch, TensorFlow, JAX), **license**, and **size**. Each model has a **model card** with performance metrics, intended use, and limitations.

Key filters when choosing a model for production:
- **Task**: matches your use case (text-classification, summarization, question-answering, ...)
- **Size**: smaller models are faster; larger models are more capable
- **License**: Apache 2.0, MIT (permissive) vs. restricted licenses (some LLaMA variants)
- **Downloads / likes**: community validation signal

In [4]:
from huggingface_hub import list_models

# List top 5 models for text-classification, sorted by downloads
models = list_models(
    filter="text-classification",
    sort="downloads",
    direction=-1,
    limit=5,
)

print("Top 5 text-classification models by downloads:\n")
for m in models:
    print(f"  {m.id:50s}  downloads: {m.downloads:>10,}")

Top 5 text-classification models by downloads:

  cross-encoder/ms-marco-MiniLM-L6-v2                 downloads: 14,517,421
  ProsusAI/finbert                                    downloads:  6,195,495
  BAAI/bge-reranker-v2-m3                             downloads:  4,609,333
  cardiffnlp/twitter-roberta-base-sentiment-latest    downloads:  4,110,123
  facebook/bart-large-mnli                            downloads:  3,796,583


In [5]:
# List top 5 models for summarization
models_summ = list_models(
    filter="summarization",
    sort="downloads",
    direction=-1,
    limit=5,
)

print("Top 5 summarization models by downloads:\n")
for m in models_summ:
    print(f"  {m.id:50s}  downloads: {m.downloads:>10,}")

Top 5 summarization models by downloads:

  google-t5/t5-base                                   downloads:  2,208,843
  facebook/bart-large-cnn                             downloads:  2,203,963
  google-t5/t5-small                                  downloads:  1,843,138
  google-t5/t5-3b                                     downloads:    820,467
  sshleifer/distilbart-cnn-12-6                       downloads:    757,288


In [6]:
# Inspect a specific model's metadata
from huggingface_hub import model_info

info = model_info(CONFIG["sentiment_model"])

print(f"Model:    {info.id}")
print(f"Pipeline: {info.pipeline_tag}")
print(f"License:  {info.card_data.get('license', 'not specified') if info.card_data else 'N/A'}")
print(f"Tags:     {info.tags[:8]}")
print(f"Library:  {info.library_name}")
print(f"Downloads: {info.downloads:,}")

Model:    distilbert/distilbert-base-uncased-finetuned-sst-2-english
Pipeline: text-classification
License:  apache-2.0
Tags:     ['transformers', 'pytorch', 'tf', 'rust', 'onnx', 'safetensors', 'distilbert', 'text-classification']
Library:  transformers
Downloads: 3,439,717


### Loading a model for inference

Two approaches:

1. **`pipeline()`** — high-level, task-oriented (recommended for quick prototyping)
2. **`AutoModel` + `AutoTokenizer`** — full control (recommended for production / custom logic)

Both download the model from the Hub on first use and cache it locally.

In [7]:
from transformers import pipeline

# High-level: pipeline("task", model="model-name")
classifier = pipeline("sentiment-analysis", model=CONFIG["sentiment_model"])

result = classifier("The quarterly report exceeded expectations across all divisions.")
print("Pipeline result:", result)

Device set to use cpu


Pipeline result: [{'label': 'POSITIVE', 'score': 0.9954332709312439}]


In [8]:
# Same result with AutoModel + AutoTokenizer (full control)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained(CONFIG["sentiment_model"])
model = AutoModelForSequenceClassification.from_pretrained(CONFIG["sentiment_model"])

text = "The quarterly report exceeded expectations across all divisions."
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
predicted_class = torch.argmax(logits, dim=-1).item()
label = model.config.id2label[predicted_class]
score = torch.softmax(logits, dim=-1)[0][predicted_class].item()

print(f"Text:  {text}")
print(f"Label: {label}  (score: {score:.4f})")

Text:  The quarterly report exceeded expectations across all divisions.
Label: POSITIVE  (score: 0.9954)


In [9]:
# Compare: pipeline vs AutoModel produce the same result
print("Pipeline approach:  ", classifier(text))
print(f"AutoModel approach: ['{{'label': '{label}', 'score': {score:.4f}}}']")
print("\nPipeline is a convenience wrapper around AutoModel + AutoTokenizer.")

Pipeline approach:   [{'label': 'POSITIVE', 'score': 0.9954332709312439}]
AutoModel approach: ['{'label': 'POSITIVE', 'score': 0.9954}']

Pipeline is a convenience wrapper around AutoModel + AutoTokenizer.


### Model selection for enterprise use

When choosing a model for production deployment:

| Criterion | What to check | Example |
|-----------|--------------|---------|
| License | Model card > License field | Apache 2.0 = commercial OK |
| Size | Number of parameters | distilbert-base = 66M (fast); bart-large = 406M (accurate) |
| Benchmark scores | Model card > Evaluation section | F1, accuracy on standard benchmarks |
| Community activity | Downloads, last update | High downloads + recent updates = well-maintained |
| Security | Known vulnerabilities in dependencies | Check `pip audit` after install |

In [10]:
# Quick model size check — parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: {CONFIG['sentiment_model']}")
print(f"  Total parameters:     {total_params:>12,}")
print(f"  Trainable parameters: {trainable_params:>12,}")
print(f"  Size (approx):        {total_params * 4 / 1e6:.1f} MB (float32)")

Model: distilbert-base-uncased-finetuned-sst-2-english
  Total parameters:       66,955,010
  Trainable parameters:   66,955,010
  Size (approx):        267.8 MB (float32)


## 5. Datasets on the Hugging Face Hub

The Hub hosts 100K+ datasets for NLP, vision, audio, and multimodal tasks. 

In [11]:
from datasets import load_dataset

# Load the IMDB dataset (movie review sentiment)
# Only load the first 200 rows of the test split for speed
imdb = load_dataset("imdb", split="test[:200]")

print(f"Dataset:  imdb (test split, first 200 rows)")
print(f"Features: {imdb.features}")
print(f"Columns:  {imdb.column_names}")
print(f"Rows:     {len(imdb)}")

Dataset:  imdb (test split, first 200 rows)
Features: {'text': Value(dtype='string', id=None), 'label': ClassLabel(names=['neg', 'pos'], id=None)}
Columns:  ['text', 'label']
Rows:     200


In [12]:
# Inspect one example
example = imdb[0]
print(f"Label: {example['label']}  (0=negative, 1=positive)")
print(f"Text:  {example['text'][:300]}...")

Label: 0  (0=negative, 1=positive)
Text:  I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, C...


## 6. Community and resources

| Resource | URL | Use |
|----------|-----|-----|
| Model Hub | huggingface.co/models | Search models by task, size, license |
| Dataset Hub | huggingface.co/datasets | Search datasets by task, language |
| Spaces | huggingface.co/spaces | Live demos hosted on HF infrastructure |
| Documentation | huggingface.co/docs/transformers | API reference, tutorials, guides |
| Model Cards | On each model page | Intended use, limitations, training data |

**Production note:** When using community models in enterprise environments, verify the license, scan for security vulnerabilities (dependency audit), and pin model versions (revision hash) for reproducibility.

In [ ]:
# Pin a model version using a specific revision (commit hash)
from transformers import AutoTokenizer

# Loading with a specific revision ensures reproducibility
tokenizer_pinned = AutoTokenizer.from_pretrained(
    CONFIG["sentiment_model"],
    revision="main",  # In production, use a specific commit hash
)

print(f"Tokenizer loaded: {CONFIG['sentiment_model']}")
print(f"Vocab size: {tokenizer_pinned.vocab_size}")
print(f"Model max length: {tokenizer_pinned.model_max_length}")

### How tokenization works

Tokenizers convert raw text into token IDs that the model understands. Understanding tokenization is essential for debugging, setting `max_length`, and estimating costs.

```
"The report exceeded expectations"
    │
    ▼  Tokenizer
[101, 1996, 3189, 14049, 10908, 102]


In [ ]:
# Tokenization step by step
text = "The quarterly earnings report exceeded analyst expectations for Q3."

# Step 1: Tokenize
tokens = tokenizer.tokenize(text)
print(f"Tokens ({len(tokens)}): {tokens}")

# Step 2: Convert to IDs
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"Token IDs: {token_ids}")

# Step 3: Full encoding (adds special tokens)
encoded = tokenizer(text, return_tensors="pt")
print(f"Input IDs shape: {encoded['input_ids'].shape}")
print(f"Input IDs:       {encoded['input_ids'][0].tolist()}")
print(f"Attention mask:  {encoded['attention_mask'][0].tolist()}")

In [ ]:
# Decode back to text
decoded = tokenizer.decode(encoded["input_ids"][0])
print(f"Decoded: {decoded}")

# Token count estimation (useful for max_length and cost)
texts = [
    "Short text.",
    "The quarterly earnings report exceeded analyst expectations for Q3 2024.",
    "Given the complexity of modern enterprise AI systems, organizations must carefully evaluate the trade-offs between managed API services and self-hosted open-source solutions.",
]
for t in texts:
    n_tokens = len(tokenizer.tokenize(t))
    print(f"  {n_tokens:3d} tokens | {t[:70]}")

### Batch inference pattern

Production systems process batches, not single inputs. The `pipeline` API handles batching automatically; with `AutoModel`, you batch via the tokenizer.

In [ ]:
# Batch with pipeline — pass a list
texts_batch = [
    "Revenue grew 15% year over year driven by cloud services.",
    "The product recall affected consumer confidence in the brand.",
    "Operating margins improved despite rising raw material costs.",
    "Employee satisfaction scores dropped to the lowest in five years.",
    "The merger creates the third-largest player in the European market.",
]

results_batch = classifier(texts_batch, truncation=True, max_length=512)

for text_item, res in zip(texts_batch, results_batch):
    print(f"  [{res['label']:>8s} {res['score']:.3f}]  {text_item}")

In [ ]:
# Batch with AutoModel — tokenize all at once, pad to same length
inputs_batch = tokenizer(
    texts_batch,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=512,
)

print(f"Batch input shape: {inputs_batch['input_ids'].shape}")  # (5, seq_len)

with torch.no_grad():
    outputs_batch = model(**inputs_batch)

probs = torch.softmax(outputs_batch.logits, dim=-1)

for i, text_item in enumerate(texts_batch):
    predicted = torch.argmax(probs[i]).item()
    label_name = model.config.id2label[predicted]
    score_val = probs[i][predicted].item()
    print(f"  [{label_name:>8s} {score_val:.3f}]  {text_item}")

### Device management — CPU vs GPU

Models run on CPU by default. Moving to GPU (if available) speeds up inference significantly. The `pipeline` API handles this with `device`; with `AutoModel`, use `.to(device)`.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Pipeline with explicit device
classifier_gpu = pipeline(
    "sentiment-analysis",
    model=CONFIG["sentiment_model"],
    device=0 if device == "cuda" else -1,
)

# AutoModel with explicit device
model_on_device = model.to(device)

# Inference on device
inputs_device = tokenizer("Test sentence for device check.", return_tensors="pt").to(device)
with torch.no_grad():
    out = model_on_device(**inputs_device)
print(f"Inference ran on: {device}")
print(f"Output logits device: {out.logits.device}")

## 7. Frameworks quick reference

Libraries used in Day 2 with their primary purpose, key functions, and one minimal example each.

| Library | Purpose in Day 2 | Key functions |
|---------|-----------------|---------------|
| `transformers` | Load and run pre-trained models | `pipeline()`, `AutoModel.from_pretrained()`, `AutoTokenizer.from_pretrained()` |
| `datasets` | Load and process datasets from Hub | `load_dataset()`, `.filter()`, `.map()`, `.select()` |
| `huggingface_hub` | List/download models and datasets | `list_models()`, `model_info()`, `hf_hub_download()` |
| `sentence-transformers` | Generate text embeddings | `SentenceTransformer()`, `.encode()` |
| `faiss-cpu` | Vector similarity search | `IndexFlatL2()`, `.add()`, `.search()` |
| `chromadb` | Persistent vector database | `Client()`, `.create_collection()`, `.add()`, `.query()` |